# 10 - Next Matchday Predictions

This notebook loads the saved deployment-ready models from notebooks 05, 05b, 05c, 06, and 06b.
It does not retrain any model. Instead, it reuses the best saved artifacts and applies them to the next available Bundesliga fixtures.


## 1. Imports and Setup

The notebook relies on saved deployment artifacts and the latest processed match table.
If the next Bundesliga fixtures are already present in the processed feature table, every saved model can be scored immediately. If not, the notebook falls back to the raw FBref schedule, rebuilds the core pre-match features for the next round from historical data, and then scores every model that has the inputs it needs. Only workflows that truly require unavailable external inputs, such as future market odds, remain blocked.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.betting_strategy import add_binary_edge
from src.config import PROCESSED_DATA_DIR, RAW_DATA_DIR, TABLES_DIR
from src.data_builder import normalize_team_name
from src.deployment_helpers import (
    build_canonical_team_lookup,
    build_next_matchday_feature_rows,
    discover_deployment_runs,
    infer_latest_played_cutoff,
    load_deployment_artifact,
    load_prediction_source_table,
    predict_with_deployment_artifact,
    select_next_matchday_candidates,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


## 2. Load the Deployment Registry and the Richest Prediction Source Table

The deployment registry tells us which best models are currently available for reuse.
The prediction source table is the richest processed table in the project, so it is the first place where we check whether the next Bundesliga fixtures already exist with the required model inputs.


In [2]:
deployment_registry = discover_deployment_runs()
source_df = load_prediction_source_table()

preview_cols = [
    col for col in ['date', 'season_id', 'matchday', 'home_team', 'away_team', 'home_goals', 'away_goals']
    if col in source_df.columns
]

print('Available deployment runs:', len(deployment_registry))
print('Prediction source table shape:', source_df.shape)

display(deployment_registry.sort_values('display_name'))
display(source_df[preview_cols].sort_values('date').tail(10))


Available deployment runs: 5
Prediction source table shape: (882, 279)


,run_key,display_name,artifact_path,metadata_path,best_model_name,best_feature_set,target_col,primary_metric,trained_through_date,trained_through_matchday,trained_through_season_id
0,double_poisson_binary,Double Poisson binary,C:\Users\cerve\Desktop\DP\match_prediction\out...,C:\Users\cerve\Desktop\DP\match_prediction\out...,double_poisson_alpha_0_1,None,home_win,None,2026-04-19T17:30:00,30,2025
1,double_poisson_multiclass,Double Poisson multiclass,C:\Users\cerve\Desktop\DP\match_prediction\out...,C:\Users\cerve\Desktop\DP\match_prediction\out...,double_poisson_alpha_0_1,None,target_1x2,None,2026-04-19T17:30:00,30,2025
2,ml_betting_binary,ML betting binary,C:\Users\cerve\Desktop\DP\match_prediction\out...,C:\Users\cerve\Desktop\DP\match_prediction\out...,logistic_regression,market_only,home_win,log_loss,2026-04-19T17:30:00,30,2025
3,ml_binary,ML binary,C:\Users\cerve\Desktop\DP\match_prediction\out...,C:\Users\cerve\Desktop\DP\match_prediction\out...,naive_bayes,top_20_rf,home_win,accuracy,2026-04-19T17:30:00,30,2025
4,ml_multiclass,ML multiclass,C:\Users\cerve\Desktop\DP\match_prediction\out...,C:\Users\cerve\Desktop\DP\match_prediction\out...,knn,compact_domain,target_1x2,accuracy,2026-04-19T17:30:00,30,2025


,date,season_id,home_team,away_team,home_goals,away_goals
872,2026-04-12 17:30:00,2025,Mainz 05,Freiburg,0,1
873,2026-04-17 18:30:00,2025,St. Pauli,FC Cologne,1,1
874,2026-04-18 13:30:00,2025,Bayer Leverkusen,Augsburg,1,2
875,2026-04-18 13:30:00,2025,Hoffenheim,Borussia Dortmund,2,1
876,2026-04-18 13:30:00,2025,Union Berlin,Wolfsburg,1,2
877,2026-04-18 13:30:00,2025,Werder Bremen,Hamburger SV,3,1
878,2026-04-18 16:30:00,2025,Eintracht Frankfurt,RasenBallsport Leipzig,1,3
879,2026-04-19 13:30:00,2025,Freiburg,FC Heidenheim,2,1
880,2026-04-19 15:30:00,2025,Bayern Munich,VfB Stuttgart,4,2
881,2026-04-19 17:30:00,2025,Borussia M.Gladbach,Mainz 05,1,1


## 3. Infer the Current Played-Match Cutoff

The next-matchday workflow needs a clear cutoff so that the predictions are interpreted correctly.
This section identifies the latest completed Bundesliga fixture currently present in the processed table.


In [3]:
cutoff_info = infer_latest_played_cutoff(source_df)
cutoff_summary = pd.DataFrame([cutoff_info])
display(cutoff_summary)


,latest_season_id,latest_played_date,latest_played_matchday
0,2025,2026-04-19 17:30:00,None


## 4. Check Whether the Next Matchday Is Already Present in the Processed Table

This is the preferred path for live-style scoring.
If future fixtures are already present in the processed table, the notebook keeps only the closest full upcoming Bundesliga matchday so that the output stays focused on the immediate next round.


In [4]:
processed_next_matchday = select_next_matchday_candidates(
    source_df,
    latest_season_id=cutoff_info.get('latest_season_id'),
    latest_played_date=cutoff_info.get('latest_played_date'),
)

fixture_preview_cols = [
    col for col in ['date', 'season_id', 'matchday', 'home_team', 'away_team']
    if col in processed_next_matchday.columns
]

print('Next-matchday fixtures already present in the processed table:', len(processed_next_matchday))
display(processed_next_matchday[fixture_preview_cols].head(20))


Next-matchday fixtures already present in the processed table: 0


,date,season_id,home_team,away_team


## 5. Fall Back to the Raw FBref Schedule When Needed

If the processed table does not yet contain future fixtures, the notebook tries to read the raw FBref schedule.
This fallback is useful for identifying the immediate next round and for scoring model families that can work with a minimal fixture table. The team names are harmonized back to the canonical project labels so that saved models see the same naming convention as during training.


In [5]:
def try_load_fbref_schedule():
    candidate_paths = [
        RAW_DATA_DIR / 'fbref' / 'schedule.parquet',
        RAW_DATA_DIR / 'fbref' / 'schedule.csv',
    ]
    last_error = None

    for path in candidate_paths:
        if not path.exists():
            continue
        try:
            if path.suffix == '.parquet':
                df = pd.read_parquet(path)
            else:
                df = pd.read_csv(path)
            return df, path, last_error
        except Exception as exc:
            last_error = exc

    return pd.DataFrame(), None, last_error


def build_schedule_fallback_candidates(schedule_df: pd.DataFrame, cutoff_dict: dict, team_lookup: dict) -> pd.DataFrame:
    if schedule_df.empty:
        return pd.DataFrame()

    schedule = schedule_df.copy()
    if any(name is not None for name in schedule.index.names):
        schedule = schedule.reset_index()

    schedule = schedule.rename(
        columns={col: str(col).strip().lower().replace(' ', '_') for col in schedule.columns}
    )

    required = {'date', 'home_team', 'away_team'}
    missing = required.difference(schedule.columns)
    if missing:
        print(f"Raw FBref schedule is missing the fallback columns: {sorted(missing)}")
        return pd.DataFrame()

    schedule['date'] = pd.to_datetime(schedule['date'], errors='coerce')
    schedule = schedule.loc[schedule['date'].notna()].copy()

    latest_played_date = cutoff_dict.get('latest_played_date')
    if latest_played_date is not None:
        schedule = schedule.loc[schedule['date'] > latest_played_date].copy()

    if 'home_goals' in schedule.columns:
        schedule = schedule.loc[schedule['home_goals'].isna()].copy()
    if 'away_goals' in schedule.columns:
        schedule = schedule.loc[schedule['away_goals'].isna()].copy()

    if schedule.empty:
        return schedule

    def canonicalize_team_name(name):
        normalized = normalize_team_name(name)
        if pd.isna(normalized):
            return name
        return team_lookup.get(normalized, name)

    schedule['home_team'] = schedule['home_team'].map(canonicalize_team_name)
    schedule['away_team'] = schedule['away_team'].map(canonicalize_team_name)

    if 'season_id' not in schedule.columns:
        schedule['season_id'] = cutoff_dict.get('latest_season_id')

    if 'week' in schedule.columns:
        schedule['matchday'] = pd.to_numeric(schedule['week'], errors='coerce')
    else:
        schedule['matchday'] = pd.NA

    if 'round' in schedule.columns:
        fallback_matchday = pd.to_numeric(
            schedule['round'].astype(str).str.extract(r'(\d+)')[0],
            errors='coerce',
        )
        schedule['matchday'] = schedule['matchday'].fillna(fallback_matchday)

    if 'game_id' not in schedule.columns:
        schedule = schedule.reset_index(drop=True)
        schedule['game_id'] = [f'schedule_{idx + 1}' for idx in range(len(schedule))]

    keep_cols = [
        col for col in ['date', 'season_id', 'game_id', 'home_team', 'away_team', 'round', 'week', 'matchday']
        if col in schedule.columns
    ]
    return schedule[keep_cols].sort_values(['date', 'game_id']).reset_index(drop=True)


team_lookup = build_canonical_team_lookup(source_df)
raw_schedule_df, raw_schedule_path, raw_schedule_error = try_load_fbref_schedule()
raw_schedule_candidates = build_schedule_fallback_candidates(raw_schedule_df, cutoff_info, team_lookup)
schedule_fallback = select_next_matchday_candidates(
    raw_schedule_candidates,
    latest_season_id=cutoff_info.get('latest_season_id'),
    latest_played_date=cutoff_info.get('latest_played_date'),
)

if raw_schedule_path is not None:
    print(f'Raw schedule source detected: {raw_schedule_path.name}')
if raw_schedule_error is not None:
    print(f'Last raw schedule load error: {raw_schedule_error}')

print('Fallback next-matchday fixtures available from the raw schedule:', len(schedule_fallback))
display(schedule_fallback.head(20))


Raw schedule source detected: schedule.parquet
Fallback next-matchday fixtures available from the raw schedule: 9


,date,season_id,game_id,home_team,away_team,round,week,matchday
0,2026-04-24,2025,None,RasenBallsport Leipzig,Union Berlin,<NA>,31,31
1,2026-04-25,2025,None,Augsburg,Eintracht Frankfurt,<NA>,31,31
2,2026-04-25,2025,None,Hamburger SV,Hoffenheim,<NA>,31,31
3,2026-04-25,2025,None,FC Heidenheim,St. Pauli,<NA>,31,31
4,2026-04-25,2025,None,FC Cologne,Bayer Leverkusen,<NA>,31,31
5,2026-04-25,2025,None,Mainz 05,Bayern Munich,<NA>,31,31
6,2026-04-25,2025,None,Wolfsburg,Borussia M.Gladbach,<NA>,31,31
7,2026-04-26,2025,None,Dortmund,Freiburg,<NA>,31,31
8,2026-04-26,2025,None,VfB Stuttgart,Werder Bremen,<NA>,31,31


## 6. Select the Fixture Input That Will Be Scored

The notebook first prefers the processed feature table because it is the only source that can feed every saved model family directly.
If that is not available yet, the raw schedule fallback is enriched with rebuilt pre-match features from the historical match table so that the standard ML and double Poisson workflows can still be scored. In both cases, the selected fixture table is restricted to the immediate next Bundesliga matchday only.


In [6]:
if not processed_next_matchday.empty:
    candidate_fixtures = processed_next_matchday.copy()
    fixture_source = 'processed feature table'
elif not schedule_fallback.empty:
    candidate_fixtures = build_next_matchday_feature_rows(source_df, schedule_fallback)
    fixture_source = 'raw FBref schedule enriched with rebuilt historical features'
else:
    candidate_fixtures = pd.DataFrame()
    fixture_source = None

selected_matchday = None
if not candidate_fixtures.empty and 'matchday' in candidate_fixtures.columns and candidate_fixtures['matchday'].notna().any():
    selected_matchday = int(pd.to_numeric(candidate_fixtures['matchday'], errors='coerce').dropna().min())

candidate_summary = pd.DataFrame(
    [
        {
            'fixture_source': fixture_source,
            'n_next_matchday_fixtures': len(candidate_fixtures),
            'selected_matchday': selected_matchday,
            'latest_season_id': cutoff_info.get('latest_season_id'),
            'latest_played_matchday': cutoff_info.get('latest_played_matchday'),
            'latest_played_date': cutoff_info.get('latest_played_date'),
        }
    ]
)

display(candidate_summary)

if candidate_fixtures.empty:
    print(
        'No next-matchday fixtures are currently available. Update the raw schedule and rerun notebooks 02 and 03b if you want the full ML stack to score the next round.'
    )
else:
    preview_cols = [col for col in ['date', 'season_id', 'matchday', 'home_team', 'away_team', 'home_rest_days', 'away_rest_days', 'home_elo_pre', 'away_elo_pre'] if col in candidate_fixtures.columns]
    display(candidate_fixtures[preview_cols])


C:\Users\cerve\Desktop\DP\match_prediction\src\advanced_features.py:338: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"diff_{f}"] = df[f"home_{f}"] - df[f"away_{f}"]


,fixture_source,n_next_matchday_fixtures,selected_matchday,latest_season_id,latest_played_matchday,latest_played_date
0,raw FBref schedule enriched with rebuilt histo...,9,31,2025,None,2026-04-19 17:30:00


,date,season_id,matchday,home_team,away_team,home_rest_days,away_rest_days,home_elo_pre,away_elo_pre
0,2026-04-24,2025,31,RasenBallsport Leipzig,Union Berlin,6.0,6.0,1611.083379,1440.188224
1,2026-04-25,2025,31,Augsburg,Eintracht Frankfurt,6.0,6.0,1485.126114,1537.792908
2,2026-04-25,2025,31,Hamburger SV,Hoffenheim,6.0,6.0,1463.302620,1538.264649
3,2026-04-25,2025,31,FC Heidenheim,St. Pauli,6.0,6.0,1375.245268,1420.107892
4,2026-04-25,2025,31,FC Cologne,Bayer Leverkusen,6.0,6.0,1431.938372,1640.807254
5,2026-04-25,2025,31,Mainz 05,Bayern Munich,6.0,6.0,1512.415776,1770.640924
6,2026-04-25,2025,31,Wolfsburg,Borussia M.Gladbach,6.0,6.0,1406.108356,1454.011120
7,2026-04-26,2025,31,Dortmund,Freiburg,6.0,6.0,1500.000000,1532.808307
8,2026-04-26,2025,31,VfB Stuttgart,Werder Bremen,6.0,6.0,1597.763245,1466.316883


## 7. Score the Candidate Fixtures with the Saved Deployment Models

Each run is loaded from the deployment folder and scored separately.
If some runs still fail, the notebook keeps the successful predictions and records the reason for every blocked run. After the fallback feature rebuild, the most common remaining blocker should be missing future market odds for the market-aware betting workflow.


In [7]:
prediction_frames = []
prediction_failures = []

if candidate_fixtures.empty:
    all_next_predictions = pd.DataFrame()
    prediction_failures_df = pd.DataFrame()
else:
    for _, registry_row in deployment_registry.sort_values('display_name').iterrows():
        run_key = registry_row['run_key']
        try:
            artifact, metadata = load_deployment_artifact(run_key)
            pred_df = predict_with_deployment_artifact(run_key, artifact, candidate_fixtures)
            pred_df['prediction_source'] = fixture_source
            pred_df['trained_through_matchday'] = registry_row.get('trained_through_matchday')
            pred_df['trained_through_date'] = registry_row.get('trained_through_date')

            if 'benchmark_home_prob' in pred_df.columns and 'p_home_win_model' in pred_df.columns:
                pred_df = add_binary_edge(
                    pred_df,
                    model_prob_col='p_home_win_model',
                    market_prob_col='benchmark_home_prob',
                    edge_col_name='home_win_edge',
                )
            if 'benchmark_away_not_lose_prob' in pred_df.columns and 'p_away_not_lose_model' in pred_df.columns:
                pred_df = add_binary_edge(
                    pred_df,
                    model_prob_col='p_away_not_lose_model',
                    market_prob_col='benchmark_away_not_lose_prob',
                    edge_col_name='away_not_lose_edge',
                )

            prediction_frames.append(pred_df)
        except Exception as exc:
            prediction_failures.append(
                {
                    'run_key': run_key,
                    'display_name': registry_row.get('display_name'),
                    'best_model_name': registry_row.get('best_model_name'),
                    'error': str(exc),
                }
            )

    all_next_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
    prediction_failures_df = pd.DataFrame(prediction_failures)

if not all_next_predictions.empty:
    prediction_key_cols = [
        col for col in ['run_key', 'season_id', 'matchday', 'date', 'home_team', 'away_team']
        if col in all_next_predictions.columns
    ]
    if prediction_key_cols:
        all_next_predictions = all_next_predictions.drop_duplicates(subset=prediction_key_cols).copy()

print('Successful deployment runs:', 0 if all_next_predictions.empty else all_next_predictions['run_key'].nunique())
print('Scored fixtures in output:', 0 if all_next_predictions.empty else len(all_next_predictions))
if not prediction_failures_df.empty:
    display(prediction_failures_df)
else:
    print('All currently available deployment runs were scored successfully.')


Successful deployment runs: 4
Scored fixtures in output: 36


,run_key,display_name,best_model_name,error
0,ml_betting_binary,ML betting binary,logistic_regression,Cannot score run 'ml_betting_binary' because c...


## 8. Build Clean Prediction Views

The prediction tables below are intentionally deduplicated to one row per match and saved run.
This keeps the notebook readable even if the raw schedule source contains repeated rows or if the fallback feature rebuild produces overlapping intermediate rows.


In [8]:
def prediction_label(row: pd.Series):
    value = row.get('y_pred')
    if pd.isna(value):
        return pd.NA
    if row.get('target_col') == 'home_win':
        return 'Home win' if int(value) == 1 else 'Away not lose'
    return str(value)


if all_next_predictions.empty:
    multiclass_prediction_view = pd.DataFrame()
    binary_prediction_view = pd.DataFrame()
    print('No predictions were generated yet.')
else:
    all_next_predictions['prediction_label'] = all_next_predictions.apply(prediction_label, axis=1)

    multiclass_cols = [
        col for col in [
            'date', 'matchday', 'home_team', 'away_team', 'display_name', 'best_model_name', 'prediction_label',
            'p_home_win_model', 'p_draw_model', 'p_away_win_model'
        ]
        if col in all_next_predictions.columns
    ]
    binary_cols = [
        col for col in [
            'date', 'matchday', 'home_team', 'away_team', 'display_name', 'best_model_name', 'prediction_label',
            'p_home_win_model', 'p_away_not_lose_model', 'benchmark_home_prob', 'benchmark_away_not_lose_prob',
            'home_win_edge', 'away_not_lose_edge'
        ]
        if col in all_next_predictions.columns
    ]

    multiclass_prediction_view = all_next_predictions.loc[
        all_next_predictions['run_key'].isin(['ml_multiclass', 'double_poisson_multiclass']),
        multiclass_cols,
    ].sort_values(['date', 'home_team', 'display_name']).reset_index(drop=True)

    binary_prediction_view = all_next_predictions.loc[
        all_next_predictions['run_key'].isin(['ml_binary', 'ml_betting_binary', 'double_poisson_binary']),
        binary_cols,
    ].sort_values(['date', 'home_team', 'display_name']).reset_index(drop=True)

    print('Multiclass prediction view')
    display(multiclass_prediction_view)
    print('Binary prediction view')
    display(binary_prediction_view)


Multiclass prediction view


,date,matchday,home_team,away_team,display_name,best_model_name,prediction_label,p_home_win_model,p_draw_model,p_away_win_model
0,2026-04-24,31,RasenBallsport Leipzig,Union Berlin,Double Poisson multiclass,double_poisson_alpha_0_1,H,0.545774,0.225864,0.228362
1,2026-04-24,31,RasenBallsport Leipzig,Union Berlin,ML multiclass,knn,D,0.199717,0.469645,0.330637
2,2026-04-25,31,Augsburg,Eintracht Frankfurt,Double Poisson multiclass,double_poisson_alpha_0_1,A,0.365418,0.238426,0.396156
3,2026-04-25,31,Augsburg,Eintracht Frankfurt,ML multiclass,knn,A,0.135315,0.405996,0.458689
4,2026-04-25,31,FC Cologne,Bayer Leverkusen,Double Poisson multiclass,double_poisson_alpha_0_1,A,0.274869,0.229002,0.496128
5,2026-04-25,31,FC Cologne,Bayer Leverkusen,ML multiclass,knn,D,0.131837,0.467585,0.400578
6,2026-04-25,31,FC Heidenheim,St. Pauli,Double Poisson multiclass,double_poisson_alpha_0_1,H,0.400742,0.253091,0.346167
7,2026-04-25,31,FC Heidenheim,St. Pauli,ML multiclass,knn,A,0.137817,0.401334,0.460849
8,2026-04-25,31,Hamburger SV,Hoffenheim,Double Poisson multiclass,double_poisson_alpha_0_1,H,0.412012,0.232734,0.355254
9,2026-04-25,31,Hamburger SV,Hoffenheim,ML multiclass,knn,A,0.135278,0.405301,0.459421


Binary prediction view


,date,matchday,home_team,away_team,display_name,best_model_name,prediction_label,p_home_win_model,p_away_not_lose_model
0,2026-04-24,31,RasenBallsport Leipzig,Union Berlin,Double Poisson binary,double_poisson_alpha_0_1,Home win,0.545774,0.454226
1,2026-04-24,31,RasenBallsport Leipzig,Union Berlin,ML binary,naive_bayes,Away not lose,0.000061,0.999939
2,2026-04-25,31,Augsburg,Eintracht Frankfurt,Double Poisson binary,double_poisson_alpha_0_1,Away not lose,0.365418,0.634582
3,2026-04-25,31,Augsburg,Eintracht Frankfurt,ML binary,naive_bayes,Away not lose,0.000012,0.999988
4,2026-04-25,31,FC Cologne,Bayer Leverkusen,Double Poisson binary,double_poisson_alpha_0_1,Away not lose,0.274869,0.725131
5,2026-04-25,31,FC Cologne,Bayer Leverkusen,ML binary,naive_bayes,Away not lose,0.000003,0.999997
6,2026-04-25,31,FC Heidenheim,St. Pauli,Double Poisson binary,double_poisson_alpha_0_1,Away not lose,0.400742,0.599258
7,2026-04-25,31,FC Heidenheim,St. Pauli,ML binary,naive_bayes,Away not lose,0.000013,0.999987
8,2026-04-25,31,Hamburger SV,Hoffenheim,Double Poisson binary,double_poisson_alpha_0_1,Away not lose,0.412012,0.587988
9,2026-04-25,31,Hamburger SV,Hoffenheim,ML binary,naive_bayes,Away not lose,0.000010,0.999990


## 9. Build a Match-Centric Summary Table

The second prediction output is match-centric.
It pivots the run-level predictions into one table per upcoming match from the immediate next round so that the notebook stays focused on the actionable short-term forecast.


In [9]:
if all_next_predictions.empty:
    match_centric_summary = pd.DataFrame()
else:
    summary_index = [col for col in ['date', 'matchday', 'home_team', 'away_team'] if col in all_next_predictions.columns]

    prediction_pivot = (
        all_next_predictions
        .pivot_table(index=summary_index, columns='display_name', values='prediction_label', aggfunc='first')
        .sort_index(axis=1)
    )
    prediction_pivot.columns = [f'{col} pick' for col in prediction_pivot.columns]

    home_prob_pivot = (
        all_next_predictions
        .pivot_table(index=summary_index, columns='display_name', values='p_home_win_model', aggfunc='first')
        .sort_index(axis=1)
    )
    if not home_prob_pivot.empty:
        home_prob_pivot.columns = [f'{col} P(home win)' for col in home_prob_pivot.columns]

    match_centric_summary = prediction_pivot.join(home_prob_pivot, how='outer').reset_index()

print('Match-centric prediction summary')
display(match_centric_summary)


Match-centric prediction summary


,date,matchday,home_team,away_team,Double Poisson binary pick,Double Poisson multiclass pick,ML binary pick,ML multiclass pick,Double Poisson binary P(home win),Double Poisson multiclass P(home win),ML binary P(home win),ML multiclass P(home win)
0,2026-04-24,31,RasenBallsport Leipzig,Union Berlin,Home win,H,Away not lose,D,0.545774,0.545774,0.000061,0.199717
1,2026-04-25,31,Augsburg,Eintracht Frankfurt,Away not lose,A,Away not lose,A,0.365418,0.365418,0.000012,0.135315
2,2026-04-25,31,FC Cologne,Bayer Leverkusen,Away not lose,A,Away not lose,D,0.274869,0.274869,0.000003,0.131837
3,2026-04-25,31,FC Heidenheim,St. Pauli,Away not lose,H,Away not lose,A,0.400742,0.400742,0.000013,0.137817
4,2026-04-25,31,Hamburger SV,Hoffenheim,Away not lose,H,Away not lose,A,0.412012,0.412012,0.000010,0.135278
5,2026-04-25,31,Mainz 05,Bayern Munich,Away not lose,A,Away not lose,A,0.247084,0.247084,0.000001,0.065082
6,2026-04-25,31,Wolfsburg,Borussia M.Gladbach,Away not lose,H,Away not lose,A,0.418870,0.418870,0.000013,0.137242
7,2026-04-26,31,Dortmund,Freiburg,Away not lose,H,Away not lose,A,0.443427,0.443427,0.000014,0.135437
8,2026-04-26,31,VfB Stuttgart,Werder Bremen,Home win,H,Away not lose,D,0.532289,0.532289,0.000048,0.136373


## 10. Save the Next-Matchday Outputs

The saved files make the next-round predictions reusable for later reporting.
This is also helpful when the project is run repeatedly during an ongoing season because the notebook then acts as a lightweight inference layer on top of the already trained deployment artifacts, always focused on the immediate next matchday only.


In [10]:
NEXT_MATCHDAY_DIR = PROCESSED_DATA_DIR / 'next_matchday_predictions'
NEXT_MATCHDAY_TABLES_DIR = TABLES_DIR / 'next_matchday_predictions'
NEXT_MATCHDAY_DIR.mkdir(parents=True, exist_ok=True)
NEXT_MATCHDAY_TABLES_DIR.mkdir(parents=True, exist_ok=True)

deployment_registry.to_csv(NEXT_MATCHDAY_DIR / 'deployment_registry.csv', index=False)
cutoff_summary.to_csv(NEXT_MATCHDAY_DIR / 'cutoff_summary.csv', index=False)
candidate_summary.to_csv(NEXT_MATCHDAY_DIR / 'candidate_summary.csv', index=False)

if not processed_next_matchday.empty:
    processed_next_matchday.to_csv(NEXT_MATCHDAY_DIR / 'processed_next_matchday_fixtures.csv', index=False)
if not schedule_fallback.empty:
    schedule_fallback.to_csv(NEXT_MATCHDAY_DIR / 'raw_schedule_next_matchday_fixtures.csv', index=False)
if not candidate_fixtures.empty:
    candidate_fixtures.to_csv(NEXT_MATCHDAY_DIR / 'selected_candidate_fixtures.csv', index=False)
if not all_next_predictions.empty:
    all_next_predictions.to_csv(NEXT_MATCHDAY_DIR / 'all_next_predictions.csv', index=False)
if not multiclass_prediction_view.empty:
    multiclass_prediction_view.to_csv(NEXT_MATCHDAY_TABLES_DIR / 'multiclass_prediction_view.csv', index=False)
if not binary_prediction_view.empty:
    binary_prediction_view.to_csv(NEXT_MATCHDAY_TABLES_DIR / 'binary_prediction_view.csv', index=False)
if not match_centric_summary.empty:
    match_centric_summary.to_csv(NEXT_MATCHDAY_TABLES_DIR / 'match_centric_summary.csv', index=False)
if not prediction_failures_df.empty:
    prediction_failures_df.to_csv(NEXT_MATCHDAY_DIR / 'prediction_failures.csv', index=False)

print(f'Saved next-matchday outputs to: {NEXT_MATCHDAY_DIR}')
print(f'Saved table exports to: {NEXT_MATCHDAY_TABLES_DIR}')


Saved next-matchday outputs to: C:\Users\cerve\Desktop\DP\match_prediction\data\processed\next_matchday_predictions
Saved table exports to: C:\Users\cerve\Desktop\DP\match_prediction\outputs\tables\next_matchday_predictions
